In [1]:
import pandas as pd
import numpy as np
import os

import plotly.figure_factory as ff
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage


In [2]:
base_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/data/"

doc_sent = pd.read_csv(
    os.path.join(base_path, "output/DOC_SENT.csv"),
    sep="|"
)

lib = pd.read_csv(
    os.path.join(base_path, "output/LIB.csv"),
    sep="|"
)

In [3]:
doc_sent.head()

,book_number,chapter,verse,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,positive,negative,token_count
0,1,1,1,0.000000,0.100000,0.0,0.100000,0.100000,0.000000,0.000000,0.100000,0.100000,0.000000,10
1,1,1,2,0.034483,0.034483,0.0,0.068966,0.034483,0.034483,0.000000,0.034483,0.068966,0.034483,29
2,1,1,3,0.000000,0.090909,0.0,0.090909,0.090909,0.000000,0.000000,0.090909,0.090909,0.000000,11
3,1,1,4,0.058824,0.176471,0.0,0.176471,0.176471,0.058824,0.058824,0.176471,0.176471,0.058824,17
4,1,1,5,0.045455,0.045455,0.0,0.090909,0.045455,0.045455,0.000000,0.045455,0.045455,0.045455,22


In [4]:
lib.head()

,book_number,total_chapters,total_verses,total_characters,total_words,book_name,genre_id,genre_name,testament,avg_chars_per_verse,avg_words_per_verse
0,1,50,1533,195262,38265,Genesis,1,Law,Old Testament,127.372472,24.960861
1,2,40,1213,168113,32684,Exodus,1,Law,Old Testament,138.592745,26.944765
2,3,27,859,125992,24543,Leviticus,1,Law,Old Testament,146.672875,28.571595
3,4,36,1288,174297,32895,Numbers,1,Law,Old Testament,135.323758,25.539596
4,5,34,959,145443,28351,Deuteronomy,1,Law,Old Testament,151.661105,29.563087


In [5]:
emotion_cols = [
    "joy", "trust", "anticipation", "surprise",
    "anger", "fear", "disgust", "sadness"
]

# Merge book_name into doc_sent first
doc_sent_named = doc_sent.merge(
    lib[["book_number", "book_name"]],
    on="book_number",
    how="left"
)

book_emotions = (
    doc_sent_named
    .groupby("book_name")[emotion_cols]
    .mean()
    .reset_index()
    .merge(lib[["book_name", "book_number"]], on="book_name")
    .sort_values("book_number")
)

book_emotions.head()

,book_name,joy,trust,anticipation,surprise,anger,fear,disgust,sadness,book_number
29,Genesis,0.022535,0.035766,0.018713,0.006596,0.008352,0.017280,0.010655,0.010256,1
25,Exodus,0.012993,0.036855,0.016544,0.004941,0.012557,0.017968,0.018988,0.010912,2
45,Leviticus,0.015400,0.052104,0.012896,0.006017,0.016402,0.025085,0.044706,0.020123,3
53,Numbers,0.011737,0.046609,0.014214,0.005332,0.011990,0.016544,0.025588,0.013618,4
21,Deuteronomy,0.029398,0.057065,0.028758,0.007938,0.016568,0.032389,0.028863,0.014697,5


In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(book_emotions[emotion_cols])

In [7]:
Z = linkage(X_scaled, method="ward")

fig = ff.create_dendrogram(
    X_scaled,
    labels=book_emotions["book_name"].tolist(),
    linkagefun=lambda x: Z,
    orientation="bottom"
)

fig.update_layout(
    width=1400,
    height=700,
    title="Hierarchical Clustering of Books by Emotional Profile",
    xaxis_title="Book",
    yaxis_title="Distance"
)

fig.show()

This dendrogram was created by first computing the average emotional profile for each book across eight NRC emotion categories: joy, trust, anticipation, surprise, anger, fear, disgust, and sadness. Each book is therefore represented as an eight-dimensional vector of mean emotion scores. Because the scales of these emotions differ slightly, the features were standardized using z-scores so that no single emotion would dominate the clustering. Hierarchical clustering was then performed using Ward’s method, which groups books in a way that minimizes within-cluster variance. The dendrogram visualizes how books merge into clusters based on their emotional similarity, with the vertical height representing distance in standardized emotional space.

The tree reveals two major emotional blocs. On the left, shown in green, most Old Testament narrative and historical books cluster together. Importantly, Matthew, Mark, Luke, John, and Acts also fall into this green cluster. This means the Gospels are emotionally more similar to Old Testament narrative material than to the later New Testament letters. Their emotional averages tend to be more moderate and balanced, rather than elevated in positive affect.

On the right, shown in red, most of the New Testament epistles cluster together. These books are emotionally distinct from both the Old Testament and the Gospels. This matches what we observed earlier in the canonical-order scatter plots: the epistles tend to have higher levels of joy, anticipation, and trust. The Pauline letters in particular cluster closely, suggesting a consistent emotional signature across them.

The key conclusion is that the primary emotional division in the canon is not Old Testament versus New Testament. Instead, it is narrative books, including the Gospels and Acts, versus epistolary material. The emotional shift occurs after Acts, where the later letters move into a higher positive-affect profile. The clustering analysis confirms this structural break using unsupervised methods rather than visual inspection alone.

In [8]:
graph_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs"

save_file = os.path.join(
    graph_path,
    "25_riff_3_hierarchical_clustering_emotional_profiles.html"
)

fig.write_html(save_file)

print("Saved to:", save_file)

Saved to: /Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs/25_riff_3_hierarchical_clustering_emotional_profiles.html
